# Fine-tuning LoRA - Modele medical (Phi-3.5 Instruct)

POC experimental. Dataset : `medical_train.json` / `medical_val.json` (prepares par DATA).


## GPU


In [ ]:
!nvidia-smi


## Dependances


In [ ]:
# Versions compatibles avec l'environnement Colab actuel (CUDA 12.x / triton 3.x)
!pip -q install -U "bitsandbytes>=0.45.0" "transformers>=4.46" "peft>=0.14" "accelerate>=1.2" "datasets>=3.0"
# NB: apres cette cellule -> Execution > Redemarrer la session, puis reprendre a 'Donnees'.


## Donnees


In [ ]:
import json, os
from google.colab import files

if not (os.path.exists('medical_train.json') and os.path.exists('medical_val.json')):
    files.upload()

with open('medical_train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open('medical_val.json', encoding='utf-8') as f:
    val_raw = json.load(f)

# Limite pour accelerer l'entrainement (mettre None pour tout garder)
MAX_TRAIN, MAX_VAL = 2000, 200
if MAX_TRAIN:
    train_raw = train_raw[:MAX_TRAIN]
if MAX_VAL:
    val_raw = val_raw[:MAX_VAL]

len(train_raw), len(val_raw)


## Modele (QLoRA 4-bit)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)


## Configuration LoRA


In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## Preparation des donnees


In [ ]:
from datasets import Dataset

MAX_LEN = 384  # troncature ; padding dynamique via le collator

def to_text(ex):
    messages = [
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list(train_raw).map(to_text)
val_ds = Dataset.from_list(val_raw).map(to_text)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tok, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tok, batched=True, remove_columns=val_ds.column_names)


## Entrainement


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='phi35_medical_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='epoch',
    save_total_limit=1,
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()


## Metriques


In [ ]:
import json
hist = trainer.state.log_history

train_pts = [(h['step'], h['loss']) for h in hist if 'loss' in h]
eval_pts = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]

for s, l in train_pts:
    print(f'step {s:5d} | train_loss {l:.4f}')
for s, l in eval_pts:
    print(f'step {s:5d} | val_loss   {l:.4f}')

with open('metrics.json', 'w') as f:
    json.dump({'train_loss': train_pts, 'eval_loss': eval_pts,
               'epochs': args.num_train_epochs}, f, indent=2)


## Courbe des pertes


In [ ]:
import matplotlib.pyplot as plt
if train_pts:
    plt.plot(*zip(*train_pts), label='train loss')
if eval_pts:
    plt.plot(*zip(*eval_pts), label='val loss', marker='o')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend()
plt.title('Fine-tuning medical Phi-3.5'); plt.show()


## Inference


In [ ]:
def ask(question, max_new_tokens=256):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            use_cache=False,                     # contourne le bug DynamicCache.seen_tokens
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

ask('I have had a sore throat and mild fever for 3 days. What should I do?')


## Export de l'adapter


In [ ]:
model.save_pretrained('phi35_medical_lora')
tokenizer.save_pretrained('phi35_medical_lora')
!cp metrics.json phi35_medical_lora/ 2>/dev/null
!zip -r phi35_medical_lora.zip phi35_medical_lora > /dev/null

from google.colab import files
files.download('phi35_medical_lora.zip')
